In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler ,OneHotEncoder
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,classification_report,roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from xgboost import XGBClassifier


In [2]:
df=pd.read_csv("business_loan_dataset.csv")
df1=pd.read_csv("education_loan_dataset.csv")
df2=pd.read_csv("home_loan_dataset.csv")
df3=pd.read_csv("personal_loan_dataset.csv")
df4=pd.read_csv("vehicle_loan_dataset.csv")

In [53]:
df1.head()

,Application_ID,Age,Education_Level,Course_Type,Course_Duration,Institution_Type,Institution_Location,Admission_Status,Annual_Course_Fee,Total_Education_Cost,Loan_Amount,Family_Monthly_Income,Family_Existing_EMI,Number_of_Dependents,Previous_Academic_Performance,Co_Applicant_Occupation,Co_Applicant_Monthly_Income,Loan_Approval
0,EL100000,21,Professional Course,Computer Science,4,University,Semi Urban,Confirmed,252000,1153000,558000,32800,5500,4,Excellent,Salaried,37900,Approved
1,EL100001,21,Postgraduate,Engineering,5,Private,Semi Urban,Confirmed,212000,1183000,1100000,52600,15900,4,Average,Not Applicable,0,Rejected
2,EL100002,18,Undergraduate,Medical,4,Government,Urban,Confirmed,480000,2084000,1569000,44500,5600,2,Good,Salaried,23600,Rejected
3,EL100003,17,Undergraduate,Law,5,Private,Urban,Confirmed,204000,1097000,894000,75100,17800,4,Excellent,Private Employee,38100,Rejected
4,EL100004,23,Diploma,Engineering,3,College,Semi Urban,Confirmed,179000,668000,523000,13800,1200,4,Good,Salaried,28000,Rejected


In [54]:
df1.isnull().sum()

Application_ID                   0
Age                              0
Education_Level                  0
Course_Type                      0
Course_Duration                  0
Institution_Type                 0
Institution_Location             0
Admission_Status                 0
Annual_Course_Fee                0
Total_Education_Cost             0
Loan_Amount                      0
Family_Monthly_Income            0
Family_Existing_EMI              0
Number_of_Dependents             0
Previous_Academic_Performance    0
Co_Applicant_Occupation          0
Co_Applicant_Monthly_Income      0
Loan_Approval                    0
dtype: int64

In [55]:
print("Duplicate rows:", df1.duplicated().sum())

Duplicate rows: 0


In [56]:
print(df1.columns.tolist())

['Application_ID', 'Age', 'Education_Level', 'Course_Type', 'Course_Duration', 'Institution_Type', 'Institution_Location', 'Admission_Status', 'Annual_Course_Fee', 'Total_Education_Cost', 'Loan_Amount', 'Family_Monthly_Income', 'Family_Existing_EMI', 'Number_of_Dependents', 'Previous_Academic_Performance', 'Co_Applicant_Occupation', 'Co_Applicant_Monthly_Income', 'Loan_Approval']


In [67]:
print(df1['Loan_Approval'].value_counts())
print("\nPercentage:")
print(df1['Loan_Approval'].value_counts(normalize=True) * 100)

Loan_Approval
Approved    1136
Rejected     864
Name: count, dtype: int64

Percentage:
Loan_Approval
Approved    56.8
Rejected    43.2
Name: proportion, dtype: float64


In [68]:
X = df1.drop(columns=['Application_ID', 'Loan_Approval'])
y = df1['Loan_Approval']

In [69]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (1600, 16)
X_test: (400, 16)
y_train: (1600,)
y_test: (400,)


In [70]:
numerical_features = [
    'Age',
    'Course_Duration',
    'Annual_Course_Fee',
    'Total_Education_Cost',
    'Loan_Amount',
    'Family_Monthly_Income',
    'Family_Existing_EMI',
    'Number_of_Dependents',
    'Co_Applicant_Monthly_Income'
]

categorical_features = [
    'Education_Level',
    'Course_Type',
    'Institution_Type',
    'Institution_Location',
    'Admission_Status',
    'Previous_Academic_Performance',
    'Co_Applicant_Occupation'
]



In [71]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

In [72]:

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    
    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ),
    
    "SVM": SVC(
        probability=True,
        random_state=42
    ),
    
    "XGBoost": XGBClassifier(
        n_estimators=200,
        random_state=42,
        eval_metric='logloss'
    )
}

In [73]:
pipelines = {}

for name, model in models.items():
    pipelines[name] = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

In [74]:
pipelines

{'Logistic Regression': Pipeline(steps=[('preprocessor',
                  ColumnTransformer(transformers=[('num', StandardScaler(),
                                                   ['Age', 'Course_Duration',
                                                    'Annual_Course_Fee',
                                                    'Total_Education_Cost',
                                                    'Loan_Amount',
                                                    'Family_Monthly_Income',
                                                    'Family_Existing_EMI',
                                                    'Number_of_Dependents',
                                                    'Co_Applicant_Monthly_Income']),
                                                  ('cat',
                                                   OneHotEncoder(handle_unknown='ignore'),
                                                   ['Education_Level',
                                        

In [65]:
# Train the four models
for name, pipeline in pipelines.items():
    if name != "XGBoost":
        pipeline.fit(X_train, y_train)
        print(f"{name} trained successfully")

ValueError: could not convert string to float: 'Excellent'

In [19]:
# Encode target for XGBoost
y_xgb = y.map({
    'Rejected': 0,
    'Approved': 1
})

pipelines['XGBoost'].fit(
    X_train,
    y_xgb.loc[X_train.index]
)

print("XGBoost trained successfully")

XGBoost trained successfully


In [20]:
for name, pipeline in pipelines.items():
    y_pred = pipeline.predict(X_test)

    print(f"\n{name}")
    print("-" * 40)
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1 Score :", f1_score(y_test, y_pred))


Logistic Regression
----------------------------------------
Accuracy : 0.7125


ValueError: pos_label=1 is not a valid label. It should be one of ['Approved', 'Rejected']

In [75]:
# 1. Encode target
y = y.map({
    'Rejected': 0,
    'Approved': 1
})

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 3. Train all models
for name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    print(f"{name} trained successfully")

Logistic Regression trained successfully
Decision Tree trained successfully
Random Forest trained successfully
SVM trained successfully
XGBoost trained successfully


In [76]:
for name, pipeline in pipelines.items():
    y_pred = pipeline.predict(X_test)

    print(f"\n{name}")
    print("-" * 40)
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1 Score :", f1_score(y_test, y_pred))


Logistic Regression
----------------------------------------
Accuracy : 0.7175
Precision: 0.7435897435897436
Recall   : 0.7665198237885462
F1 Score : 0.754880694143167

Decision Tree
----------------------------------------
Accuracy : 0.625
Precision: 0.6711111111111111
Recall   : 0.6651982378854625
F1 Score : 0.668141592920354

Random Forest
----------------------------------------
Accuracy : 0.7075
Precision: 0.7182539682539683
Recall   : 0.7973568281938326
F1 Score : 0.755741127348643

SVM
----------------------------------------
Accuracy : 0.715
Precision: 0.7383966244725738
Recall   : 0.7709251101321586
F1 Score : 0.7543103448275862

XGBoost
----------------------------------------
Accuracy : 0.66
Precision: 0.6842105263157895
Recall   : 0.7444933920704846
F1 Score : 0.7130801687763713


In [77]:
from sklearn.metrics import confusion_matrix

for name, pipeline in pipelines.items():
    y_pred = pipeline.predict(X_test)

    print(f"\n{name}")
    print(confusion_matrix(y_test, y_pred))


Logistic Regression
[[113  60]
 [ 53 174]]

Decision Tree
[[ 99  74]
 [ 76 151]]

Random Forest
[[102  71]
 [ 46 181]]

SVM
[[111  62]
 [ 52 175]]

XGBoost
[[ 95  78]
 [ 58 169]]


In [24]:
# parameter tuining
from sklearn.model_selection import GridSearchCV

In [78]:
rf_params = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [None, 5, 10, 20],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4]
}


In [79]:
rf_grid = GridSearchCV(
    estimator=pipelines['Random Forest'],
    param_grid=rf_params,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train, y_train)

Fitting 5 folds for each of 108 candidates, totalling 540 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [None, 5, ...], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...], 'model__n_estimators': [100, 200, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computatio

In [80]:
print("Best Parameters:")
print(rf_grid.best_params_)

print("\nBest CV F1 Score:")
print(rf_grid.best_score_)

Best Parameters:
{'model__max_depth': 20, 'model__min_samples_leaf': 2, 'model__min_samples_split': 2, 'model__n_estimators': 100}

Best CV F1 Score:
0.7591975837286318


In [81]:
best_rf = rf_grid.best_estimator_

y_pred_rf = best_rf.predict(X_test)

print("Tuned Random Forest")
print("-" * 40)
print("Accuracy :", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall   :", recall_score(y_test, y_pred_rf))
print("F1 Score :", f1_score(y_test, y_pred_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

Tuned Random Forest
----------------------------------------
Accuracy : 0.695
Precision: 0.7075098814229249
Recall   : 0.788546255506608
F1 Score : 0.7458333333333333

Confusion Matrix:
[[ 99  74]
 [ 48 179]]


In [29]:
xgb_params = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [3, 5, 7],
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__subsample': [0.8, 1.0]
}

xgb_grid = GridSearchCV(
    estimator=pipelines['XGBoost'],
    param_grid=xgb_params,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(X_train, y_train)

print("Best Parameters:")
print(xgb_grid.best_params_)

print("\nBest CV F1 Score:")
print(xgb_grid.best_score_)

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best Parameters:
{'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__n_estimators': 300, 'model__subsample': 0.8}

Best CV F1 Score:
0.7665155906398173


In [30]:
best_xgb = xgb_grid.best_estimator_

y_pred_xgb = best_xgb.predict(X_test)

print("Tuned XGBoost")
print("-" * 40)
print("Accuracy :", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall   :", recall_score(y_test, y_pred_xgb))
print("F1 Score :", f1_score(y_test, y_pred_xgb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

Tuned XGBoost
----------------------------------------
Accuracy : 0.69
Precision: 0.7049180327868853
Recall   : 0.7678571428571429
F1 Score : 0.7350427350427351

Confusion Matrix:
[[104  72]
 [ 52 172]]


In [31]:
rf_pipeline = pipelines["Random Forest"]

In [32]:
rf_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [33]:
import pickle
with open("business_loan_model.pkl", "wb") as file:
    pickle.dump(rf_pipeline, file)

print("Business Loan model saved successfully!")

Business Loan model saved successfully!


In [82]:
lr_params = {
    'model__C': [0.01, 0.1, 1, 10, 100],
    'model__solver': ['liblinear', 'lbfgs']
}

lr_grid = GridSearchCV(
    estimator=pipelines['Logistic Regression'],
    param_grid=lr_params,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

lr_grid.fit(X_train, y_train)

print("Best Parameters:", lr_grid.best_params_)
print("Best CV F1:", lr_grid.best_score_)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best Parameters: {'model__C': 0.1, 'model__solver': 'liblinear'}
Best CV F1: 0.7624207057052395


In [83]:
lr_best = lr_grid.best_estimator_

y_pred = lr_best.predict(X_test)

print("Tuned Logistic Regression")
print("----------------------------------------")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Tuned Logistic Regression
----------------------------------------
Accuracy : 0.7125
Precision: 0.7352941176470589
Recall   : 0.7709251101321586
F1 Score : 0.7526881720430108

Confusion Matrix:
[[110  63]
 [ 52 175]]


In [84]:
final_model = pipelines["Random Forest"]

final_model.fit(X_train, y_train)

with open("education_loan_model.pkl", "wb") as file:
    pickle.dump(final_model, file)

print("Education Loan model saved successfully!")

Education Loan model saved successfully!
